<div dir="rtl" style="text-align:right"><h1>שיעור 3, חלק 1: מנוירון לייצוג – איך מחשב לומד – שלב C: ייצוגים: מילים כמספרים</h1>
<p><b>המחברת הזו היא לשלב C («ייצוגים: מילים כמספרים») בשיעור באתר הקורס – ורק לו.</b> את ההסבר קוראים באתר; כאן מריצים ומשנים.</p>
<p>מריצים תא אחרי תא בלחיצה על ▶, מלמעלה למטה. מעבד רגיל (<code>CPU</code>) מספיק; אין צורך ב-<code>GPU</code>, במפתח <code>API</code> או במנוי. <b>אין חובת הגשת מחברת.</b> המחברת מורידה מודל ייצוגים פתוח של כ-470MB. אם מופיעה אזהרה על <code>HF_TOKEN</code> – אפשר להתעלם ממנה: המודל ציבורי.</p></div>

In [ ]:
# Setup. Everything below is preinstalled on Google Colab.
import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers

print("transformers", transformers.__version__, "| torch", torch.__version__, "- setup done")


def rtl(text):
    # matplotlib draws left to right; reversing a Hebrew word displays it correctly.
    return text[::-1] if any("\u0590" <= ch <= "\u05ff" for ch in text) else text

<div dir="rtl" style="text-align:right"><h2>3 | ייצוגים (embeddings): מילים כמספרים</h2>
<p>רשת עובדת רק עם מספרים. לכן כל מילה מקבלת <b>ייצוג</b>: רשימה ארוכה של מספרים, שנלמדת באימון. למילים שמופיעות בהקשרים דומים יש ייצוגים קרובים. נשתמש במודל הייצוגים הפתוח <code>sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2</code> – אותו מודל שבמפה באתר. הוא אומן גם על זוגות של משפטים ותרגומם, ולכן מילה ותרגומה יוצאות קרובות מאוד. לכל טקסט הוא נותן 384 מספרים.</p>
<p><b>דמיון קוסינוס</b> משווה שתי רשימות: ככל שהמספר גבוה יותר – הן דומות יותר (1 – כיוון זהה). גם מילים לא קשורות מקבלות כאן מספר (בערך 0.3–0.5), ולכן משווים מספרים זה לזה ולא לאפס.</p>
<blockquote>① נחשו: מי הכי קרוב ל«לחם» – «bread», «עוגה», «קפה» או «שמח»?</blockquote></div>

In [ ]:
from transformers import AutoModel, AutoTokenizer

EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb_tok = AutoTokenizer.from_pretrained(EMB_MODEL)
emb_model = AutoModel.from_pretrained(EMB_MODEL)
emb_model.eval()


def embed(texts):
    batch = emb_tok(texts, padding=True, return_tensors="pt")
    with torch.no_grad():
        out = emb_model(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1).float()
    return ((out * mask).sum(1) / mask.sum(1)).numpy()  # mean pooling: one vector per text


def cosine(a, b):
    va, vb = embed([a, b])
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))


print("numbers per word:", embed(["לחם"]).shape[1])
for a, b in [("לחם", "bread"), ("לחם", "עוגה"), ("לחם", "קפה"), ("לחם", "שמח"), ("מחר", "tomorrow"), ("שמח", "עצוב")]:
    print(f"{a:>6} ~ {b:<8}  cosine = {cosine(a, b):.2f}")

<div dir="rtl" style="text-align:right"><p>④ «שמח» ו«עצוב» הפוכים במשמעות, ובכל זאת קרובים למדי – שתיהן מילים של רגש שמופיעות בהקשרים דומים. ייצוגים משקפים <b>הקשרים שבהם מילים מופיעות</b>, לא הגדרה של משמעות.</p>
<h3>3א | המילים שלכם</h3>
<p>③ כתבו ברשימה מילים משלכם, בעברית או באנגלית (8–20 מילים). התא ימצא לכל מילה את שתי השכנות הקרובות לה <b>מתוך הרשימה שלכם</b>, ויצייר מפה בשני ממדים.</p>
<blockquote>① לפני ההרצה: אילו מילים לדעתכם יתקבצו יחד?</blockquote></div>

In [ ]:
my_words = ["לחם", "עוגה", "עוגיות", "bread", "cake", "קפה", "תה", "coffee", "tea",
            "מחר", "אתמול", "tomorrow", "שמח", "עצוב", "happy", "אמא", "אבא", "mother"]  # <- ③ your own words

vectors = embed(my_words)
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
sim = unit @ unit.T
for i, w in enumerate(my_words):
    order = [j for j in np.argsort(-sim[i]) if j != i][:2]
    print(f"{w:>9}: " + ", ".join(f"{my_words[j]} ({sim[i, j]:.2f})" for j in order))

centered = vectors - vectors.mean(0)
_, singular, vt = np.linalg.svd(centered, full_matrices=False)
xy = centered @ vt[:2].T
print(f"\nthe 2D map keeps {(singular[:2] ** 2).sum() / (singular ** 2).sum():.0%} of the variance")
plt.figure(figsize=(7, 5.5))
plt.scatter(xy[:, 0], xy[:, 1], color="#4a6c7a")
for (x, y), w in zip(xy, my_words):
    plt.annotate(rtl(w), (x, y), textcoords="offset points", xytext=(4, 4))
plt.title("PCA to 2D - distances are approximate")
plt.show()

<div dir="rtl" style="text-align:right"><p>④ <b>המפה דוחסת</b> 384 מספרים לשניים, ולכן המרחקים בה משוערים. השכנות שהודפסו מעל המפה מחושבות במלוא 384 המספרים – בהן סומכים.</p></div>

<div dir="rtl" style="text-align:right"><p><b>סיימתם את המחברת של שלב C.</b> חזרו ללשונית של אתר הקורס וענו שם על שאלות השלב.</p></div>